### Import


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_fscore_support,
)
import re
import math
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import signal
import subprocess
import tempfile

import multiprocessing
import traceback

from typing import List, Dict, Any, Tuple, Callable, Optional, Union
import pprint as p

### Load Datasets


In [ ]:
# df = pd.read_parquet("../../datasets/train-00000-of-00004-ee77a7de79eb2ab2.parquet")
# df.shape
# df.to_json("../../datasets/core/code-search-net-python-1.json", lines=True, orient="records")

In [ ]:
mbpp_df = pd.read_json("../../datasets/core/sanitized-mbpp.json")
print(mbpp_df.shape)
print(mbpp_df.columns)
# p.pprint(mbpp_df.iloc[0]['code'])

In [ ]:
# classEval = pd.read_parquet("../../datasets/classEval.parquet")
# print(classEval.shape)
# print(classEval.columns)
# p.pprint(classEval.iloc[88]['solution_code'])

# with open('test.py', 'w') as f:
#     f.write(classEval.iloc[35]['solution_code'])

In [ ]:
# humanevalplus_df = pd.read_json(
#     "../../datasets/core/humanevalplus-python-1000.json", lines=True
# )
# print(humanevalplus_df.shape)
# print(humanevalplus_df.columns)
# p.pprint(humanevalplus_df.iloc[5]['code'])

# with open('test.py', 'w') as f:
#     f.write(humanevalplus_df.iloc[25]['code'])

In [ ]:
# humaneval_df = pd.read_parquet("../../datasets/core/human_eval_164.parquet")
# print(humaneval_df.shape)
# print(humaneval_df.columns)
# # p.pprint(humaneval_df.iloc[2]['original_string'])

# with open('human_test.py', 'w') as f:
#     f.write(humaneval_df.iloc[111]['prompt'])

In [ ]:
# csn_df = pd.read_parquet("../../datasets/train-00000-of-00004-ee77a7de79eb2ab2.parquet")
# print(csn_df.shape)
# print(csn_df.columns)
# # p.pprint(csn_df.iloc[2]['code'])

In [ ]:
# hmcorp_df = pd.read_json("../../datasets/core/redefined_hmcorp.jsonl", lines=True)
# print(hmcorp_df.shape)
# print(hmcorp_df.columns)

In [ ]:
# hmcorp_human_df = hmcorp_df[["id", "human_code"]].copy()
# hmcorp_ai_df = hmcorp_df[["id", "ai_code"]].copy()

# # rename to 'code' and reset index (drop the old index)
# hmcorp_human_df = hmcorp_human_df.rename(columns={"human_code": "code"}).reset_index(
#     drop=True
# )
# hmcorp_ai_df = hmcorp_ai_df.rename(columns={"ai_code": "code"}).reset_index(drop=True)

# print(hmcorp_human_df.columns)
# print(hmcorp_ai_df.shape)

### Helper Methods


In [ ]:
import keyword

avoided_tokens = [
    "self",
    "append",
    "join",
    "dummy_function",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
]

RESERVED_TOKENS = [
    "self",
    "re",
    "cls",
    "append",
    "join",
    "dummy_function",
    "find",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
]

COMMON_METHODS = {
    "lower",
    "upper",
    "strip",
    "split",
    "replace",
    "endswith",
    "startswith",
    "append",
    "extend",
    "insert",
    "remove",
    "pop",
    "sort",
    "clear",
    "keys",
    "values",
    "items",
    "get",
    "update",
    "copy",
    "format",
    "join",
    "count",
    "index",
}
BUILTIN_NAMES = set(dir(builtins)).union(COMMON_METHODS).union(RESERVED_TOKENS)

def extract_identifiers_from_code(code: str):
    """
    Extract all identifiers (variable, function, class, argument names)
    from a given Python code string using AST parsing, skipping builtins.
    """
    identifiers = []

    class IdentifierVisitor(ast.NodeVisitor):
        def visit_Name(self, node):
            if node.id not in BUILTIN_NAMES:
                identifiers.append(node.id)
            self.generic_visit(node)

        def visit_FunctionDef(self, node):
            if node.name not in BUILTIN_NAMES:
                identifiers.append(node.name)
            for arg in node.args.args:
                if arg.arg not in BUILTIN_NAMES:
                    identifiers.append(arg.arg)
            self.generic_visit(node)

        def visit_ClassDef(self, node):
            if node.name not in BUILTIN_NAMES:
                identifiers.append(node.name)
            self.generic_visit(node)

        def visit_Attribute(self, node):
            if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_METHODS:
                identifiers.append(node.attr)
            self.generic_visit(node)

        # Ignore imports completely
        def visit_Import(self, node):
            pass

        def visit_ImportFrom(self, node):
            pass

    try:
        tree = ast.parse(code)
        IdentifierVisitor().visit(tree)
    except Exception:
        return []

    return identifiers

COMMON_STD_METHODS = {
    "self",
    "re",
    "cls",
    "append",
    "join",
    "dummy_function",
    "find",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
    "now",
    "today",
    "timedelta",
    "strptime",
    "date",
    "time",
    "datetime",
    "logging",
    "log",
    "info",
    "debug",
    "error",
    "warning",
    "exception",
    "lower",
    "upper",
    "strip",
    "split",
    "replace",
    "endswith",
    "startswith",
    "append",
    "extend",
    "insert",
    "remove",
    "pop",
    "sort",
    "clear",
    "keys",
    "values",
    "items",
    "get",
    "update",
    "copy",
    "format",
    "join",
    "count",
    "index",
}

BUILTIN_NAMES = set(dir(builtins)).union(COMMON_STD_METHODS)

class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()
        self.docstrings = []

    def visit_FunctionDef(self, node):
        name = node.name

        # classify
        if name.startswith("__") and name.endswith("__"):
            pass
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # add arguments
        for arg in node.args.args:
            if arg.arg not in BUILTIN_NAMES:
                self.non_public_vars.add(arg.arg)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "function_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.non_public_vars.add(node.name)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "class_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id not in BUILTIN_NAMES:
                if target.id.isupper():
                    self.public_vars.add(target.id)
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # only include custom attributes
        if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
            self.non_public_vars.add(node.attr)
        self.generic_visit(node)

    def visit_Name(self, node):
        if node.id not in BUILTIN_NAMES:
            self.non_public_vars.add(node.id)
        self.generic_visit(node)

    def visit_Module(self, node):
        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "module_docstring",
                    "name": "__main__",
                    "line_number": getattr(node, "lineno", 1),
                    "content": doc,
                }
            )
        self.generic_visit(node)


In [ ]:
def get_tokens(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return set()

    visitor = CodeNavigator()
    visitor.visit(tree)

    all_tokens = (
        visitor.public_classes
        # | visitor.public_funcs
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.non_public_classes
    )

    # 🚫 Remove known stdlib or logging-related names
    cleaned_tokens = {
        t for t in all_tokens if t not in COMMON_STD_METHODS and t not in BUILTIN_NAMES
    }

    return cleaned_tokens

### Testing Detection


In [ ]:
# load sample ai generated code

codes = []
codebase_path = "../../output/gemini_exp1-1_during_gen_v3_100_mbpp"
for root, dirs, files in os.walk(codebase_path):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            with open(file_path, "r", encoding="utf-8") as f:
                code = f.read()
                codes.append(code)

len(codes)

In [ ]:
# code = ""
# with open("test.py", "r") as f:
#     code = f.read()

In [ ]:
# sample_count = 5
# sample_codes = codes[sample_count + 10 : sample_count + 20]
# # for code in sample_codes:
# identifiers = extract_identifiers_from_code(code)
# tokens = get_tokens(code)
# # print("CODE: \n", code)
# print("-----")
# print("IDENTIFIERS: \n", set(identifiers))
# print("TOKENS: \n", tokens)
# missed = set(identifiers) - set(tokens)
# print("MISSED IDENTIFIERS: ", *set(missed), sep="\n")

### Token Extraction & Prompt Generation



In [ ]:
from collections import defaultdict
from pathlib import Path
import json

# ============================================================================
# MULTI-LLM CONFIGURATION
# ============================================================================
# Add new models here by specifying their name and output directory path
# Example: "claude": "../../output/claude_exp3-1_during_gen_v1_100_mbpp"

MODEL_PATHS = {
    "gemini_best": "../../output/gemini_exp3-1_during_gen_v1_100_mbpp",
    # "gemini_natural": "../../output/gemini_exp5_during_gen_v1_100_mbpp", 
    # "gemini_reverse": "../../output/gemini_exp3-2_during_gen_v2_100_mbpp", 
    # "gemini_reversev2": "../../output/gemini_exp3-2_during_gen_v3_100_mbpp", 
    "claude_mbpp": "../../output/claude_exp3-1_during_gen_v1_100_mbpp",
    "claude_humaneval": "../../output/claude_exp4_during_gen_v1_164_humaneval",
    "human_mbpp": "../../datasets/core/sanitized-mbpp-sample-100-codes",
    "human_humaneval": "../../datasets/core/humaneval-164-codes",
    
}

def load_codes_from_dir(directories):
    codes = []
    if isinstance(directories, str):
        directories = [directories]
    
    for directory in directories:
        if not Path(directory).exists():
            print(f"⚠️ Directory not found: {directory}")
            continue
        
        for file in Path(directory).glob("*.py"):
            try:
                with open(file, "r", encoding="utf-8") as f:
                    codes.append(f.read())
            except:
                pass
    return codes

# Load all model codes dynamically
model_codes = {}
for model_name, model_path in MODEL_PATHS.items():
    model_codes[model_name] = load_codes_from_dir(model_path)
    print(f"✅ Loaded {len(model_codes[model_name])} {model_name.upper()} codes from {model_path}")

#! For backward compatibility
qwen_codes = model_codes.get("qwen", [])
gemini_codes = model_codes.get("gemini", [])

best_codes = qwen_codes if len(qwen_codes) > 0 else gemini_codes
print(f"\n🎯 Using {len(best_codes)} best codes for analysis")
print(f"📊 Models configured: {list(MODEL_PATHS.keys())}")


In [ ]:
letter_tokens = defaultdict(Counter)

# Dynamically create letter_tokens for each model
model_letter_tokens = {}
for model_name in MODEL_PATHS.keys():
    model_letter_tokens[model_name] = defaultdict(Counter)

# Extract tokens for each model
for model_name, codes in model_codes.items():
    print(f"\n📍 Processing {model_name.upper()}...")
    for code in codes:
        identifiers = extract_identifiers_from_code(code)
        
        for ident in identifiers:
            if len(ident) > 1 and ident not in avoided_tokens:
                first_letter = ident[0].lower()
                if first_letter.isalpha():
                    model_letter_tokens[model_name][first_letter][ident] += 1
                    letter_tokens[first_letter][ident] += 1  # Combined

    unique_count = sum(len(v) for v in model_letter_tokens[model_name].values())
    print(f"   ✅ Extracted {unique_count} unique tokens from {model_name}")

# Print summary
print(f"\n📊 SUMMARY:")
print(f"✅ Total unique tokens (combined): {sum(len(v) for v in letter_tokens.values())}")
for model_name in MODEL_PATHS.keys():
    unique_count = sum(len(v) for v in model_letter_tokens[model_name].values())
    print(f"✅ {model_name.upper()}: {unique_count} unique tokens")

# For backward compatibility
qwen_letter_tokens = model_letter_tokens.get("qwen", defaultdict(Counter))
gemini_letter_tokens = model_letter_tokens.get("gemini", defaultdict(Counter))

letter_top_tokens = {}
for letter in sorted(letter_tokens.keys()):
    top_10 = letter_tokens[letter].most_common(10)
    letter_top_tokens[letter] = top_10


In [ ]:
import os

output_dir = Path('token_analysis_gemini_final')
output_dir.mkdir(exist_ok=True)

In [ ]:
natural_tokens_dict = {}
natural_tokens_with_freq = {}

# Save combined tokens
for letter in sorted(letter_tokens.keys()):
    top_10 = letter_tokens[letter].most_common(10)
    natural_tokens_dict[letter] = [token for token, _ in top_10]
    natural_tokens_with_freq[letter] = [{"token": token, "freq": freq} for token, freq in top_10]

with open(output_dir / "natural_tokens.json", "w") as f:
    json.dump(natural_tokens_dict, f, indent=2)

with open(output_dir / "natural_tokens_with_freq.json", "w") as f:
    json.dump(natural_tokens_with_freq, f, indent=2)

print("✅ Saved natural_tokens.json and natural_tokens_with_freq.json (combined)")

# Save model-specific tokens
print(f"\n💾 Saving model-specific token files...")
for model_name in MODEL_PATHS.keys():
    model_tokens_dict = {}
    model_tokens_with_freq = {}
    
    for letter in sorted(model_letter_tokens[model_name].keys()):
        top_10 = model_letter_tokens[model_name][letter].most_common(10)
        model_tokens_dict[letter] = [token for token, _ in top_10]
        model_tokens_with_freq[letter] = [{"token": token, "freq": freq} for token, freq in top_10]
    
    # Save both formats
    with open(output_dir / f"{model_name}_tokens.json", "w") as f:
        json.dump(model_tokens_dict, f, indent=2)
    
    with open(output_dir / f"{model_name}_tokens_with_freq.json", "w") as f:
        json.dump(model_tokens_with_freq, f, indent=2)
    
    print(f"   ✅ Saved {model_name}_tokens.json and {model_name}_tokens_with_freq.json")


In [ ]:
summary_stats = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "total_codes_analyzed": len(best_codes),
    "models_configured": list(MODEL_PATHS.keys()),
    "model_code_counts": {model: len(codes) for model, codes in model_codes.items()},
    "total_unique_tokens_combined": sum(len(v) for v in letter_tokens.values()),
    "model_unique_token_counts": {
        model: sum(len(v) for v in model_letter_tokens[model].values())
        for model in MODEL_PATHS.keys()
    },
    "letter_coverage_combined": len(letter_tokens),
    "letter_coverage_per_model": {
        model: len(model_letter_tokens[model])
        for model in MODEL_PATHS.keys()
    },
}

with open(output_dir / "analysis_summary.json", "w") as f:
    json.dump(summary_stats, f, indent=2)

print("✅ Saved analysis_summary.json")
print(f"\n📊 Summary Statistics:")
print(json.dumps(summary_stats, indent=2))


In [ ]:
# Multi-Model Comparison Visualizations
print("📊 Generating multi-model comparison visualizations...\n")

# 1. Model Diversity Comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Get all model counters
model_names = list(MODEL_PATHS.keys())
model_all_counters = {}
model_all_tokens_sets = {}

for model_name in model_names:
    counter = Counter()
    for letter in model_letter_tokens[model_name]:
        counter.update(model_letter_tokens[model_name][letter])
    model_all_counters[model_name] = counter
    model_all_tokens_sets[model_name] = set(counter.keys())

# Plot 1: Total unique tokens per model
models_list = list(MODEL_PATHS.keys())
token_counts = [len(model_all_tokens_sets[m]) for m in models_list]
colors_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(models_list)]

axes[0].bar(models_list, token_counts, color=colors_palette, alpha=0.8, edgecolor='black')
axes[0].set_ylabel('Number of Unique Tokens', fontsize=11, fontweight='bold')
axes[0].set_title('Total Unique Tokens by Model', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(token_counts):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=10)

# Plot 2: Average token frequency per model
avg_freqs = []
for model_name in models_list:
    if len(model_all_counters[model_name]) > 0:
        avg_freq = np.mean(list(model_all_counters[model_name].values()))
        avg_freqs.append(avg_freq)
    else:
        avg_freqs.append(0)

axes[1].bar(models_list, avg_freqs, color=colors_palette, alpha=0.8, edgecolor='black')
axes[1].set_ylabel('Average Token Frequency', fontsize=11, fontweight='bold')
axes[1].set_title('Average Token Frequency by Model', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(avg_freqs):
    axes[1].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(output_dir / "multi_model_comparison.png", dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved multi_model_comparison.png")


In [ ]:
# 2. Token Overlap Heatmap (Pairwise Comparison)
if len(model_names) > 1:
    # Calculate pairwise overlaps
    overlap_matrix = np.zeros((len(model_names), len(model_names)))
    
    for i, model1 in enumerate(model_names):
        for j, model2 in enumerate(model_names):
            if i == j:
                overlap_matrix[i, j] = len(model_all_tokens_sets[model1])
            else:
                overlap = len(model_all_tokens_sets[model1] & model_all_tokens_sets[model2])
                total = len(model_all_tokens_sets[model1] | model_all_tokens_sets[model2])
                overlap_matrix[i, j] = (overlap / total * 100) if total > 0 else 0
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(overlap_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
    
    ax.set_xticks(range(len(model_names)))
    ax.set_yticks(range(len(model_names)))
    ax.set_xticklabels([m.upper() for m in model_names], fontsize=11, fontweight='bold')
    ax.set_yticklabels([m.upper() for m in model_names], fontsize=11, fontweight='bold')
    
    # Add text annotations
    for i in range(len(model_names)):
        for j in range(len(model_names)):
            text = ax.text(j, i, f'{overlap_matrix[i, j]:.1f}%' if i != j else f'{int(overlap_matrix[i, j])}',
                          ha="center", va="center", color="black", fontweight='bold', fontsize=10)
    
    ax.set_title('Token Overlap Similarity Matrix (%)', fontsize=12, fontweight='bold')
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Overlap %', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_dir / "multi_model_overlap_heatmap.png", dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved multi_model_overlap_heatmap.png")
    
    # 3. Detailed overlap statistics
    print(f"\n📊 Token Overlap Analysis:")
    print("=" * 70)
    for i, model1 in enumerate(model_names):
        for j, model2 in enumerate(model_names):
            if i < j:  # Only print unique pairs
                set1 = model_all_tokens_sets[model1]
                set2 = model_all_tokens_sets[model2]
                overlap = len(set1 & set2)
                unique1 = len(set1 - set2)
                unique2 = len(set2 - set1)
                union = len(set1 | set2)
                
                print(f"\n{model1.upper()} ↔ {model2.upper()}:")
                print(f"  Shared tokens: {overlap} ({100*overlap/union:.1f}%)")
                print(f"  {model1.upper()}-only: {unique1} ({100*unique1/union:.1f}%)")
                print(f"  {model2.upper()}-only: {unique2} ({100*unique2/union:.1f}%)")
                print(f"  Total unique: {union}")
else:
    print("⚠️ Need at least 2 models for overlap analysis. Please add more models to MODEL_PATHS.")


In [ ]:
# 4. Top N Tokens Comparison Across Models
n_top_tokens = 12
fig, axes = plt.subplots(1, len(model_names), figsize=(5*len(model_names), 6))

# Handle single model case
if len(model_names) == 1:
    axes = [axes]

for idx, model_name in enumerate(model_names):
    top_n = model_all_counters[model_name].most_common(n_top_tokens)
    tokens = [t[0] for t in top_n]
    freqs = [t[1] for t in top_n]
    
    axes[idx].barh(range(len(tokens)), freqs, color=colors_palette[idx], alpha=0.8, edgecolor='black')
    axes[idx].set_yticks(range(len(tokens)))
    axes[idx].set_yticklabels(tokens, fontsize=9)
    axes[idx].invert_yaxis()
    axes[idx].set_xlabel('Frequency', fontweight='bold')
    axes[idx].set_title(f'{model_name.upper()}: Top {n_top_tokens} Tokens', fontweight='bold')
    axes[idx].grid(axis='x', alpha=0.3)
    
    # Add frequency labels
    for i, v in enumerate(freqs):
        axes[idx].text(v + 0.5, i, str(v), va='center', fontweight='bold', fontsize=8)

plt.tight_layout()
plt.savefig(output_dir / "multi_model_top_tokens.png", dpi=100, bbox_inches='tight')
plt.show()

print(f"✅ Saved multi_model_top_tokens.png")

# Print top tokens summary
print(f"\n🔝 Top {n_top_tokens} Tokens by Model:")
print("=" * 70)
for model_name in model_names:
    top_n = model_all_counters[model_name].most_common(n_top_tokens)
    print(f"\n{model_name.upper()}:")
    for rank, (token, freq) in enumerate(top_n, 1):
        print(f"  {rank:2d}. {token:20s} → {freq:3d}")


In [ ]:
# 5. Comprehensive Multi-Model Analysis Export
multi_model_analysis = {
    "metadata": {
        "timestamp": pd.Timestamp.now().isoformat(),
        "models": model_names,
        "total_models": len(model_names),
    },
    "individual_models": {},
    "pairwise_comparisons": {}
}

# Individual model statistics
for model_name in model_names:
    counter = model_all_counters[model_name]
    tokens_set = model_all_tokens_sets[model_name]
    
    multi_model_analysis["individual_models"][model_name] = {
        "total_unique_tokens": len(tokens_set),
        "total_codes": len(model_codes[model_name]),
        "code_count": len(model_codes[model_name]),
        "avg_frequency": float(np.mean(list(counter.values()))) if counter else 0,
        "std_frequency": float(np.std(list(counter.values()))) if counter else 0,
        "max_frequency": int(counter.most_common(1)[0][1]) if counter else 0,
        "top_3_tokens": [t[0] for t in counter.most_common(3)],
        "letter_coverage": len(model_letter_tokens[model_name])
    }

# Pairwise comparisons
if len(model_names) > 1:
    for i, model1 in enumerate(model_names):
        for j, model2 in enumerate(model_names):
            if i < j:
                set1 = model_all_tokens_sets[model1]
                set2 = model_all_tokens_sets[model2]
                overlap = set1 & set2
                
                key = f"{model1}_vs_{model2}"
                multi_model_analysis["pairwise_comparisons"][key] = {
                    "shared_tokens": len(overlap),
                    f"{model1}_unique": len(set1 - set2),
                    f"{model2}_unique": len(set2 - set1),
                    "total_union": len(set1 | set2),
                    "overlap_percentage": float(100 * len(overlap) / len(set1 | set2)),
                    "shared_top_5": [t for t in counter.most_common(5) for counter in [
                        Counter({k: model_all_counters[model1][k] + model_all_counters[model2][k] 
                                for k in overlap})
                    ]]
                }

with open(output_dir / "multi_model_comprehensive_analysis.json", "w") as f:
    json.dump(multi_model_analysis, f, indent=2)

print("✅ Saved multi_model_comprehensive_analysis.json")
print(f"\n📊 Multi-Model Analysis Summary:")
print(json.dumps(multi_model_analysis["individual_models"], indent=2))

if multi_model_analysis["pairwise_comparisons"]:
    print(f"\n🔄 Pairwise Overlap Summary:")
    for key, stats in multi_model_analysis["pairwise_comparisons"].items():
        print(f"\n{key}:")
        print(f"  Shared: {stats['shared_tokens']} ({stats['overlap_percentage']:.1f}%)")
        print(f"  Unique tokens: {stats[key.split('_vs_')[0] + '_unique']} vs {stats[key.split('_vs_')[1] + '_unique']}")


In [ ]:
summary_stats = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "total_codes_analyzed": len(best_codes),
    "models_configured": list(MODEL_PATHS.keys()),
    "model_code_counts": {model: len(codes) for model, codes in model_codes.items()},
    "total_unique_tokens_combined": sum(len(v) for v in letter_tokens.values()),
    "model_unique_token_counts": {
        model: sum(len(v) for v in model_letter_tokens[model].values())
        for model in MODEL_PATHS.keys()
    },
    "letter_coverage_combined": len(letter_tokens),
    "letter_coverage_per_model": {
        model: len(model_letter_tokens[model])
        for model in MODEL_PATHS.keys()
    },
}

with open(output_dir / "analysis_summary.json", "w") as f:
    json.dump(summary_stats, f, indent=2)

print("\n=== ANALYSIS SUMMARY ===")
for model in MODEL_PATHS.keys():
    print(f"{model.upper()}: {summary_stats['model_code_counts'][model]} codes, {summary_stats['model_unique_token_counts'][model]} tokens")
print(f"Total unique tokens (combined): {summary_stats['total_unique_tokens_combined']}")
print(f"Letters covered: {summary_stats['letter_coverage_combined']}/26")

In [ ]:
freqs = natural_tokens_with_freq['a']
freqs

In [ ]:
review_data = {}
for letter in sorted(natural_tokens_dict.keys()):
    tokens = natural_tokens_dict[letter]
    freqs = natural_tokens_with_freq[letter]
    
    # Create a frequency dictionary for quick lookup since freqs is a list of dicts
    freq_dict = {item['token']: item['freq'] for item in freqs}
    review_data[letter] = [{"token": t, "frequency": freq_dict.get(t, 0)} for t in tokens]

with open(output_dir / "token_review.json", "w") as f:
    json.dump(review_data, f, indent=2)

print("✅ Saved token_review.json for manual inspection")


### Visualizations


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 5))

letters = sorted(letter_tokens.keys())
model_names_list = list(MODEL_PATHS.keys())
colors_pal = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(model_names_list)]

width = 0.8 / len(model_names_list)
x = np.arange(len(letters))

for idx, model in enumerate(model_names_list):
    # Calculate total unique tokens for this model
    total_unique = sum(len(v) for v in model_letter_tokens[model].values())
    # Calculate proportions instead of counts
    proportions = [len(model_letter_tokens[model].get(l, {})) / total_unique if total_unique > 0 else 0 for l in letters]
    ax.bar(x + idx*width - width*len(model_names_list)/2, proportions, width, label=model.upper(), alpha=0.8)

ax.set_xlabel('Starting Letter')
ax.set_ylabel('Proportion')
ax.set_title('Proportion of Unique Tokens per Starting Letter')
ax.set_xticks(x)
ax.set_xticklabels(letters)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / "token_proportions.png", dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved token_proportions.png")

In [ ]:
fig, axes = plt.subplots(len(model_names_list), 1, figsize=(10, 3*len(model_names_list)))
if len(model_names_list) == 1:
    axes = [axes]

for idx, model in enumerate(model_names_list):
    top_letters = sorted(model_letter_tokens[model].items(), key=lambda x: len(x[1]), reverse=True)[:10]
    letters_top = [l for l, _ in top_letters]
    counts_top = [len(c) for _, c in top_letters]
    
    axes[idx].barh(range(len(letters_top)), counts_top, color=colors_pal[idx], alpha=0.8)
    axes[idx].set_yticks(range(len(letters_top)))
    axes[idx].set_yticklabels([f"'{l}'" for l in letters_top])
    axes[idx].set_xlabel('Unique Token Count')
    axes[idx].set_title(f'{model.upper()}: Top 10 Starting Letters')
    axes[idx].grid(axis='x', alpha=0.3)
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.savefig(output_dir / "top_letters_diversity.png", dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved top_letters_diversity.png")

In [ ]:
fig, axes = plt.subplots(1, len(model_names_list), figsize=(8*len(model_names_list), 8))
if len(model_names_list) == 1:
    axes = [axes]

cmaps = ['YlOrRd', 'YlGnBu', 'Purples', 'Blues', 'Greens']

for idx, model in enumerate(model_names_list):
    model_heatmap = []
    letters_list = sorted(model_letter_tokens[model].keys())
    
    for letter in letters_list:
        top_3 = model_letter_tokens[model][letter].most_common(3)
        model_heatmap.append([freq for _, freq in top_3] + [0] * (3 - len(top_3)))
    
    model_heatmap = np.array(model_heatmap)
    im = axes[idx].imshow(model_heatmap, cmap=cmaps[idx % len(cmaps)], aspect='auto')
    axes[idx].set_xticks(range(3))
    axes[idx].set_xticklabels(['1st', '2nd', '3rd'])
    axes[idx].set_yticks(range(len(letters_list)))
    axes[idx].set_yticklabels(letters_list)
    axes[idx].set_xlabel('Token Rank')
    axes[idx].set_ylabel('Starting Letter')
    axes[idx].set_title(f'{model.upper()}: Top 3 Token Frequencies')
    plt.colorbar(im, ax=axes[idx], label='Frequency')

plt.tight_layout()
plt.savefig(output_dir / "token_frequency_heatmap.png", dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved token_frequency_heatmap.png")

In [ ]:
model_tokens_sets = {m: set() for m in model_names_list}
for model in model_names_list:
    for letter in model_letter_tokens[model]:
        model_tokens_sets[model].update(model_letter_tokens[model][letter].keys())

if len(model_names_list) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    letters_list = sorted(model_letter_tokens[model_names_list[0]].keys())
    x = np.arange(len(letters_list))
    width = 0.8 / len(model_names_list)
    
    all_model_freqs = {m: [] for m in model_names_list}
    
    for idx, model in enumerate(model_names_list):
        letter_overlaps = []
        for letter in letters_list:
            letter_overlaps.append(len(model_letter_tokens[model][letter]))
            all_model_freqs[model].extend(model_letter_tokens[model][letter].values())
        
        axes[0, 0].bar(x + idx*width - width*len(model_names_list)/2, letter_overlaps, width, label=model.upper(), alpha=0.8)
    
    axes[0, 0].set_xlabel('Starting Letter')
    axes[0, 0].set_ylabel('Token Count')
    axes[0, 0].set_title('Tokens per Letter by Model')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(letters_list)
    axes[0, 0].legend()
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    overlap_labels = []
    overlaps_pct = []
    for i in range(len(model_names_list)):
        for j in range(i+1, len(model_names_list)):
            m1, m2 = model_names_list[i], model_names_list[j]
            overlap = len(model_tokens_sets[m1] & model_tokens_sets[m2])
            union = len(model_tokens_sets[m1] | model_tokens_sets[m2])
            overlap_labels.append(f"{m1}-{m2}")
            overlaps_pct.append(100 * overlap / union if union > 0 else 0)
    
    if overlaps_pct:
        axes[0, 1].bar(range(len(overlaps_pct)), overlaps_pct, color=colors_pal[:len(overlaps_pct)], alpha=0.8)
        axes[0, 1].set_ylabel('Overlap %')
        axes[0, 1].set_title('Pairwise Token Overlap')
        axes[0, 1].set_xticks(range(len(overlaps_pct)))
        axes[0, 1].set_xticklabels(overlap_labels, rotation=45, ha='right')
        axes[0, 1].grid(axis='y', alpha=0.3)
    
    for idx, model in enumerate(model_names_list):
        axes[1, 0].hist(all_model_freqs[model], bins=20, alpha=0.6, label=model.upper(), edgecolor='black')
    
    axes[1, 0].set_xlabel('Token Frequency')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Token Frequency Distribution')
    axes[1, 0].legend()
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    model_stats = {}
    for model in model_names_list:
        if all_model_freqs[model]:
            model_stats[model] = {
                'avg': np.mean(all_model_freqs[model]),
                'std': np.std(all_model_freqs[model])
            }
    
    if model_stats:
        x_stats = np.arange(len(model_stats))
        width = 0.35
        avgs = [model_stats[m]['avg'] for m in model_names_list if m in model_stats]
        stds = [model_stats[m]['std'] for m in model_names_list if m in model_stats]
        
        axes[1, 1].bar(x_stats - width/2, avgs, width, label='Avg Freq', alpha=0.8)
        axes[1, 1].bar(x_stats + width/2, stds, width, label='Std Dev', alpha=0.8)
        axes[1, 1].set_ylabel('Value')
        axes[1, 1].set_title('Frequency Statistics')
        axes[1, 1].set_xticks(x_stats)
        axes[1, 1].set_xticklabels([m.upper() for m in model_names_list if m in model_stats])
        axes[1, 1].legend()
        axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / "token_overlap_statistics.png", dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved token_overlap_statistics.png")

In [ ]:
model_counters = {}
for model in model_names_list:
    counter = Counter()
    for letter in model_letter_tokens[model]:
        counter.update(model_letter_tokens[model][letter])
    model_counters[model] = counter

n_top = 15
fig, axes = plt.subplots(1, len(model_names_list), figsize=(8*len(model_names_list), 6))
if len(model_names_list) == 1:
    axes = [axes]

for idx, model in enumerate(model_names_list):
    top_n = model_counters[model].most_common(n_top)
    tokens = [t[0] for t in top_n]
    freqs = [t[1] for t in top_n]
    
    axes[idx].barh(range(len(tokens)), freqs, color=colors_pal[idx], alpha=0.8)
    axes[idx].set_yticks(range(len(tokens)))
    axes[idx].set_yticklabels(tokens, fontsize=9)
    axes[idx].invert_yaxis()
    axes[idx].set_xlabel('Frequency')
    axes[idx].set_title(f'{model.upper()}: Top {n_top} Tokens')
    axes[idx].grid(axis='x', alpha=0.3)
    
    for i, v in enumerate(freqs):
        axes[idx].text(v + 0.1, i, str(v), va='center', fontweight='bold', fontsize=8)

plt.tight_layout()
plt.savefig(output_dir / "top_tokens_comparison.png", dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved top_tokens_comparison.png")
for model in model_names_list:
    top_n = model_counters[model].most_common(n_top)
    print(f"\n{model.upper()} Top {n_top}:")
    for i, (token, freq) in enumerate(top_n, 1):
        print(f"  {i:2d}. {token:20s} → {freq:3d}")

In [ ]:
comprehensive_analysis = {
    "metadata": {
        "timestamp": pd.Timestamp.now().isoformat(),
        "models": model_names_list,
        "total_models": len(model_names_list)
    },
    "individual_models": {},
    "pairwise_comparisons": {}
}

for model in model_names_list:
    freq_dist = []
    for letter in model_letter_tokens[model]:
        freq_dist.extend(model_letter_tokens[model][letter].values())
    
    counter = model_counters[model]
    tokens_set = model_tokens_sets[model]
    
    comprehensive_analysis["individual_models"][model] = {
        "total_unique_tokens": len(tokens_set),
        "code_count": len(model_codes[model]),
        "avg_frequency": float(np.mean(freq_dist)) if freq_dist else 0,
        "std_frequency": float(np.std(freq_dist)) if freq_dist else 0,
        "max_frequency": int(counter.most_common(1)[0][1]) if counter else 0,
        "top_5_tokens": [t[0] for t in counter.most_common(5)],
        "letter_coverage": len(model_letter_tokens[model])
    }

if len(model_names_list) > 1:
    for i in range(len(model_names_list)):
        for j in range(i+1, len(model_names_list)):
            m1, m2 = model_names_list[i], model_names_list[j]
            set1 = model_tokens_sets[m1]
            set2 = model_tokens_sets[m2]
            overlap = set1 & set2
            
            key = f"{m1}_vs_{m2}"
            comprehensive_analysis["pairwise_comparisons"][key] = {
                "shared_tokens": len(overlap),
                f"{m1}_unique": len(set1 - set2),
                f"{m2}_unique": len(set2 - set1),
                "total_union": len(set1 | set2),
                "overlap_percentage": float(100 * len(overlap) / len(set1 | set2)) if len(set1 | set2) > 0 else 0
            }

with open(output_dir / "comprehensive_model_analysis.json", "w") as f:
    json.dump(comprehensive_analysis, f, indent=2)

print("✅ Saved comprehensive_model_analysis.json")
for model in model_names_list:
    stats = comprehensive_analysis["individual_models"][model]
    print(f"\n{model.upper()}:")
    print(f"  Tokens: {stats['total_unique_tokens']}, Codes: {stats['code_count']}, Avg Freq: {stats['avg_frequency']:.2f}")
    print(f"  Top 5: {', '.join(stats['top_5_tokens'])}")

if comprehensive_analysis["pairwise_comparisons"]:
    print(f"\nPairwise Overlaps:")
    for key, data in comprehensive_analysis["pairwise_comparisons"].items():
        print(f"  {key}: {data['shared_tokens']} shared ({data['overlap_percentage']:.1f}%)")

### 📊 Visualization Summary

This analysis has generated **4 comprehensive visualizations**:

#### 1. **top_letters_diversity.png** - Top 10 Letters by Token Count
- Horizontal bar chart comparing Qwen vs Gemini across 10 most productive letters
- Shows which starting letters produce the most diverse vocabulary in each model
- **Insight**: Identifies letters where models differ most in token generation patterns

#### 2. **token_frequency_heatmap.png** - Top 3 Token Frequencies per Letter
- Side-by-side heatmaps (Qwen red scale, Gemini blue scale)
- Shows frequency of the 1st, 2nd, and 3rd most common tokens for each letter
- **Insight**: Visualizes frequency distribution patterns across letters and models

#### 3. **token_overlap_statistics.png** - Comprehensive Overlap Analysis (4 panels)
- **Panel 1**: Per-letter token overlap (overlap, Qwen-only, Gemini-only)
- **Panel 2**: Overall token overlap distribution
- **Panel 3**: Token frequency distribution histogram
- **Panel 4**: Frequency statistics comparison (avg, std dev)
- **Insight**: Shows which tokens are shared vs unique, and frequency variation patterns

#### 4. **top_tokens_comparison.png** - Top 15 Most Common Tokens
- Two horizontal bar charts (Qwen blue, Gemini orange)
- Displays the 15 most frequently used tokens in each model
- Frequency values labeled on each bar
- **Insight**: Identifies high-impact tokens for few-shot prompt engineering

#### 5. **comprehensive_model_analysis.json** - Statistical Summary
- Complete model comparison metrics in JSON format
- Includes diversity coefficients, coverage statistics, and top tokens
- Exportable for further analysis or paper documentation

---

### 🎯 Key Takeaways for Watermarking

1. **Model Uniqueness**: % of unique tokens shows models generate different vocabularies
2. **Frequency Patterns**: Coefficient of variation indicates consistency in token usage
3. **Overlap Analysis**: Shared tokens can form robust watermark anchors
4. **Top Tokens**: Can be prioritized in watermark detection algorithms
5. **Per-Letter Analysis**: Enables targeted watermark embedding strategies